# Module 4 — Feature Selection

**By the end of this notebook, you will be able to:**
- Explain why adding every available column to a linear model can make it *worse*, not better
- Use univariate selection (`SelectKBest`) and recursive selection (`RFE`) to pick a smaller, effective feature set
- Explain why the two methods can disagree, and what that tells you about redundant features

**Context:** Every activity so far has used the same 8 hand-picked satellite columns (`DEFAULT_COLUMNS`). The raw data actually has 70 satellite columns — the 8 you have been using, plus secondary measurements (viewing angles, alternate retrievals, sensor geometry) that were deliberately set aside until now. This module uses all 70, and lets an algorithm choose instead of a human.

## Rebuild the cleaned dataset, on the full column set

**Exercise:** Same sequence as Module 2 (`restrict_to_scope` → `columns_above_missing_threshold` → `drop_columns` → `fill_missing_by_city` → `add_temporal_features`), but scoped to `SATELLITE_COLUMNS` below instead of the 8-column `DEFAULT_COLUMNS` — everything else is identical.

In [ ]:
# TODO: rebuild enriched on the full satellite column set, reusing the
# Module 2 functions you already completed


## What if you just use all of them?

After Module 2's cleaning (drop-then-fill on this wider column set), you have 66 candidate features now, instead of 8. Try the obvious thing first: fit `LinearRegression` on all of them, and look at the fitted coefficients.

In [ ]:
# Given: fit on every candidate feature, look at the largest coefficients
model = LinearRegression()
model.fit(enriched[feature_cols], enriched["pm2_5"])

coefficients = pd.Series(model.coef_, index=feature_cols).sort_values(key=np.abs, ascending=False)
print("pm2_5 range in this data:", enriched["pm2_5"].min(), "-", enriched["pm2_5"].max())
coefficients.head(10)

## Do any of these columns say almost the same thing?

Before jumping to conclusions about *why* the coefficient above is so large, check directly: compute the correlation of every candidate feature against every **other** candidate feature (not against `pm2_5`), and look at the strongest pairs.

Two choices worth explaining before you do:
- **Pearson only, not Spearman.** Module 1 used both, because the question there was "is there a relationship at all, linear or not." Here the question is narrower: does a *linear* model's coefficient estimation get confused — a fact about the design matrix, specific to linear (Pearson) correlation.
- **No p-value.** With ~8000 rows, even a trivial correlation (r=0.1) has a p-value around 1e-16 — large samples make p-values stop being informative about anything except sample size. See [Lin, Lucas & Shmueli (2013), *Too Big to Fail: Large Samples and the p-Value Problem*](https://pubsonline.informs.org/doi/10.1287/isre.2013.0480). Only the size of `r` tells you anything useful here.

Also, unlike Module 1, no `dropna` gymnastics are needed: `enriched` was already fully cleaned two cells ago (0 missing values), so `DataFrame.corr()` can use every row for every pair.

In [ ]:
# Given: correlation of every candidate feature against every other one
corr_matrix = enriched[feature_cols].corr().abs()
upper_triangle = np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)
top_pairs = corr_matrix.where(upper_triangle).stack().sort_values(ascending=False)
top_pairs.head(10).round(4)

**Dozens of pairs above 0.9.** The strongest ones are not even the NO2 family: `uvaerosolindex_solar_azimuth_angle` and `ozone_solar_azimuth_angle` correlate at essentially **1.00**, and `cloud_cloud_top_height`/`cloud_cloud_base_height` at **0.999** — the same satellite-geometry or cloud-layer measurement, copied into different products' metadata. `nitrogendioxide_no2_column_number_density` and its `tropospheric` variant, close to 0.97, are only one example among many.

This is the same *family* of problem as `site_latitude`/`site_longitude` in Session 2 — a linear model producing a huge, unstable coefficient — but a different cause. Back then, one feature barely varied within a city and jumped to a very different value in another. Here, many features carry almost the exact same information: when two columns say almost the same thing, a linear model cannot uniquely decide how much credit to assign to each, so small changes in the data can swing the split between them wildly. This is called **multicollinearity**, and it is exactly what produced the coefficient you saw above.

See what this actually does to generalization: evaluate this same feature set with `evaluate_group_cv` (Module 3).

In [ ]:
# Given: expect this to look much worse than anything you have seen so far
result_all = evaluation.evaluate_group_cv(LinearRegression(), enriched, feature_cols, groups_col="city")
{k: v for k, v in result_all.items() if k != "folds"}

Unstable in-sample coefficients become catastrophic out-of-sample: trained on three cities, the model's huge, barely-determined coefficients get multiplied by a fourth city's slightly different values, and the error explodes far beyond anything you have seen in this course. More features is not automatically better.

[`scikit-learn`'s feature selection guide](https://scikit-learn.org/stable/modules/feature_selection.html) documents two families of methods to cut this down: **univariate** (score each feature against the target, independently) and **recursive** (fit repeatedly, dropping the least useful feature each time).

## Univariate selection: `SelectKBest`

**Exercise:** Complete `select_k_best_features` in `src/air_quality/selection.py` (new file, already scaffolded) — its docstring and `tests/test_selection.py` specify exactly what it should do. Run `uv run pytest tests/test_selection.py -v` until it passes, then apply it here with `k=8`, the same number of columns Session 2 used.

In [ ]:
# TODO: apply select_k_best_features once you have implemented it


**Look closely at the names.** Several of them end in `solar_azimuth_angle`, from *different* satellite products (sulphurdioxide, carbonmonoxide, ozone, uvaerosolindex) — exactly the kind of near-duplicate pair the correlation diagnostic above surfaced. Confirm it directly on this specific selection.

In [ ]:
# Given: correlation among the SelectKBest picks themselves
enriched[skb_selected].corr().round(2)

These four correlate at **1.00** with each other, same as before. Each one independently looks like a real predictor of `pm2_5` (it is a proxy for time of day), so `SelectKBest` happily keeps all four — spending half of its 8-column budget on one piece of information repeated four times. It has no way to know they are redundant: it never looks at more than one feature at a time.

## Recursive selection: `RFE`

**Exercise:** Complete `select_features_rfe` in the same file. Then apply it with `LinearRegression()` and `n_features_to_select=8`, and compare its picks to `SelectKBest`'s.

In [ ]:
# TODO: apply select_features_rfe once you have implemented it


In [ ]:
# Given: what do the two methods agree and disagree on?
print("kept by both:", set(skb_selected) & set(rfe_selected))
print("SelectKBest only:", set(skb_selected) - set(rfe_selected))
print("RFE only:", set(rfe_selected) - set(skb_selected))

RFE fits with all remaining features still in the model at each step: once one `solar_azimuth_angle` column is in, its near-duplicates add almost nothing more, so they tend to be the ones eliminated first — RFE's picks lean toward genuinely different pollutant measurements instead.

## Does it actually matter? Measure it

Compare all three feature sets with `evaluate_group_cv`: every candidate feature, `SelectKBest`'s 8, and RFE's 8.

In [ ]:
# Given: same evaluation, three feature sets
for name, cols in [
    ("all 66 features", feature_cols),
    ("SelectKBest (k=8)", skb_selected),
    ("RFE (k=8)", rfe_selected),
]:
    result = evaluation.evaluate_group_cv(LinearRegression(), enriched, cols, groups_col="city")
    print(f"{name}: rmse_mean={result['rmse_mean']:.2f}  r2_mean={result['r2_mean']:.3f}")

**Question:** RFE's 8 columns come close to matching (or beating) the RMSE you saw in Module 3 with the 8 *hand-picked* `DEFAULT_COLUMNS` — an algorithm rediscovered, on its own, roughly what a human chose by inspection. Why does `SelectKBest` do noticeably worse here despite also picking 8 columns? What does this tell you about when a univariate method is enough, and when it is not?

## From notebook to pipeline

Update `run_advanced` in `src/air_quality/workflows.py` again: scope to `SATELLITE_COLUMNS` (not just `DEFAULT_COLUMNS`) instead of the narrower Module 2 scope, add a feature-selection step (`select_features_rfe` is the safer default, given what you just saw) before evaluating. As with the previous modules, there is no test for this — verify it by calling it below.

In [ ]:
from air_quality.workflows import run_advanced

run_advanced()

## Wrap-up

Write down: which 8 columns RFE selected, one concrete number showing why the all-features model should not be trusted, and in one sentence why RFE handled the redundant `solar_azimuth_angle` columns better than `SelectKBest`.

_Your observations here._